# Personal AI Agent - RAG

This notebook implements a Retrieval-Augmented Generation (RAG)
system over my personal knowledge base.

## Imports

In [1]:
import os
from pathlib import Path
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_chroma import Chroma

from langchain_openai import ChatOpenAI,OpenAIEmbeddings

c:\Users\Win10\Documents\GitHub\Avatar\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

BASE_DIR = Path.cwd().parent

KNOWLEDGE_BASE_DIR = BASE_DIR / "knowledge-base"
CHROMA_DIR = BASE_DIR / "chroma_db"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

LLM_MODEL = "gpt-4.1-nano"

API_KEY = os.getenv("AVALAI_API_KEY")

BASE_URL = "https://api.avalai.ir/v1"

## Chunking

In [4]:
files = list(KNOWLEDGE_BASE_DIR.glob("*.md"))

for file in files:
    print(file.name)

Aboutme.md
Education.md
Interests.md
Projects.md
Skills.md


In [3]:
documents = []

for file_path in KNOWLEDGE_BASE_DIR.glob("*.md"):
    text = file_path.read_text(encoding="utf-8")

    doc_type = file_path.stem.lower()

    document = Document(
        page_content=text,
        metadata={
            "source": file_path.name,
            "doc_type": doc_type,
        }
    )

    documents.append(document)

In [6]:
documents

[Document(metadata={'source': 'Aboutme.md', 'doc_type': 'aboutme'}, page_content='# About Me\n\n## Who Am I?\n\nHi! I\'m Mohammad Hosein and I\'m 25 years old 👋\n\nI\'m an Electrical Engineering graduate who somehow decided that dealing with electricity wasn\'t enough and started dealing with **Artificial Intelligence** instead. 😄\n\nMy background is in Electrical Engineering - Power, but my current path is strongly focused on **AI Engineering, Machine Learning, LLMs, RAG, and Agentic AI**.\n\nI\'m basically trying to turn:\n\n`Python + Curiosity + Too Many Questions`\n\ninto:\n\n`AI Engineer 🤖`\n\n---\n\n## My Journey\n\nMy programming journey started with Python.\n\nThen things escalated quickly:\n\n```text\nPython\n   ↓\nData Science\n   ↓\nMachine Learning\n   ↓\nLLMs\n   ↓\nRAG\n   ↓\nFine-tuning\n   ↓\nAgentic AI\n   ↓\n...what\'s next? 👀\n```\n\nI enjoy understanding how things work under the hood, not just copying code and hoping it works.\n\nSometimes this means spending an un

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")

Documents: 5
Chunks: 24


In [8]:
for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i} ---")
    print("Metadata:", chunk.metadata)
    print(chunk.page_content)


--- Chunk 0 ---
Metadata: {'source': 'Aboutme.md', 'doc_type': 'aboutme'}
# About Me

## Who Am I?

Hi! I'm Mohammad Hosein and I'm 25 years old 👋

I'm an Electrical Engineering graduate who somehow decided that dealing with electricity wasn't enough and started dealing with **Artificial Intelligence** instead. 😄

My background is in Electrical Engineering - Power, but my current path is strongly focused on **AI Engineering, Machine Learning, LLMs, RAG, and Agentic AI**.

I'm basically trying to turn:

`Python + Curiosity + Too Many Questions`

into:

`AI Engineer 🤖`

---

## My Journey

My programming journey started with Python.

Then things escalated quickly:

```text
Python
   ↓
Data Science
   ↓
Machine Learning
   ↓
LLMs
   ↓
RAG
   ↓
Fine-tuning
   ↓
Agentic AI
   ↓
...what's next? 👀
```

--- Chunk 1 ---
Metadata: {'source': 'Aboutme.md', 'doc_type': 'aboutme'}
```text
Python
   ↓
Data Science
   ↓
Machine Learning
   ↓
LLMs
   ↓
RAG
   ↓
Fine-tuning
   ↓
Agentic AI
   ↓
...wha

In [5]:
from collections import Counter

chunk_distribution = Counter(
    chunk.metadata["doc_type"]
    for chunk in chunks
)

chunk_distribution

Counter({'projects': 11,
         'aboutme': 7,
         'skills': 3,
         'education': 2,
         'interests': 1})

## Embedding


In [48]:
# EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", base_url = BASE_URL, api_key = API_KEY)
# embeddings = HuggingFaceEmbeddings(
    # model_name=EMBEDDING_MODEL
# )

In [49]:
if os.path.exists(CHROMA_DIR):
    Chroma(
        persist_directory=str(CHROMA_DIR),
        embedding_function=embeddings,
        collection_name="personal_agent"
    ).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(CHROMA_DIR),
    collection_name="personal_agent"
)

In [50]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4
    }
)

In [51]:
query ="how old mohammad hosein is?"

retrieved_docs = retriever.invoke(query)

In [52]:
for i, doc in enumerate(retrieved_docs):
    print(f"\n{'=' * 60}")
    print(f"Result {i + 1}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Doc Type: {doc.metadata['doc_type']}")
    print("=" * 60)
    print(doc.page_content)


Result 1
Source: Aboutme.md
Doc Type: aboutme
# About Me

## Who Am I?

Hi! I'm Mohammad Hosein and I'm 25 years old 👋

I'm an Electrical Engineering graduate who somehow decided that dealing with electricity wasn't enough and started dealing with **Artificial Intelligence** instead. 😄

My background is in Electrical Engineering - Power, but my current path is strongly focused on **AI Engineering, Machine Learning, LLMs, RAG, and Agentic AI**.

I'm basically trying to turn:

`Python + Curiosity + Too Many Questions`

into:

`AI Engineer 🤖`

---

## My Journey

My programming journey started with Python.

Then things escalated quickly:

```text
Python
   ↓
Data Science
   ↓
Machine Learning
   ↓
LLMs
   ↓
RAG
   ↓
Fine-tuning
   ↓
Agentic AI
   ↓
...what's next? 👀
```

Result 2
Source: Aboutme.md
Doc Type: aboutme
This is not the final version of me.

It's `Mohammad v0.x` — still under development. 🛠️

---

## Where I'm Going

My current direction is:

```text
Electrical Engineering
  

## LLM and RAG function

In [56]:
llm = ChatOpenAI(
    model="gpt-4.1-nano",
    api_key=os.getenv("AVALAI_API_KEY"),
    base_url="https://api.avalai.ir/v1",
    temperature=0
)

In [57]:
prompt = ChatPromptTemplate.from_template("""
You are Mohammad Hosein Salarali's personal AI assistant.

Answer the user's question using only the information provided
in the context below.

Rules:
- Do not invent or assume personal information.
- Do not use information outside the provided context.
- If the answer cannot be found in the context, say:
  "I don't have enough information in my knowledge base."
- Keep the answer clear and concise.

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
def rag(question: str):
    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    response = llm.invoke(messages)

    sources = list({
        doc.metadata["source"]
        for doc in docs
    })

    return {
        "answer": response.content,
        "sources": sources
    }

In [63]:
questions = [
    "What did Mohammad study at university?",
    "What programming skills does Mohammad have?",
    "What AI projects has Mohammad built?",
    "What is Mohammad currently learning?",
    "What are Mohammad's hobbies?",
    "What is Mohammad's career goal?"
]

for question in questions:
    result = rag(question)

    print("=" * 70)
    print("Question:", question)
    print("Answer:", result["answer"])
    print("Sources:", result["sources"])

Question: What did Mohammad study at university?
Answer: Mohammad studied Electrical Engineering - Power at Islamic Azad University.
Sources: [{'source': 'Education.md', 'doc_type': 'education'}, {'source': 'Aboutme.md', 'doc_type': 'aboutme'}, {'source': 'Aboutme.md', 'doc_type': 'aboutme'}, {'source': 'Aboutme.md', 'doc_type': 'aboutme'}]
Question: What programming skills does Mohammad have?
Answer: Mohammad has advanced Python programming skills, including Object-Oriented Programming, Error Handling, Functions, Modules, File Handling, and Virtual Environments.
Sources: [{'source': 'Aboutme.md', 'doc_type': 'aboutme'}, {'source': 'Aboutme.md', 'doc_type': 'aboutme'}, {'source': 'Education.md', 'doc_type': 'education'}, {'source': 'Skills.md', 'doc_type': 'skills'}]
Question: What AI projects has Mohammad built?
Answer: Mohammad has built a Restaurant AI Chatbot, which is an LLM application designed to work as a restaurant assistant.
Sources: [{'source': 'Aboutme.md', 'doc_type': 'abo

## Visualization

In [53]:
collection = vectorstore._collection
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]

In [54]:
# Prework
collection = vectorstore._collection
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange','yellow'][['aboutme', 'education', 'projects', 'interests','skills'].index(t)] for t in doc_types]

In [55]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42, perplexity=5)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()